In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from utils import ASSETS_DIR



In [ ]:
df = pd.read_parquet(ASSETS_DIR / 'investigator_ftldlbd_nacc74.parquet')
df.describe()

### Il dataset aveva dati longitudinali per una prima analisi sono stati tenuti solo i dati trasversali, cioè la prima visita.
### Tagliamo i dati delle versioni precedenti alla 3, che fanno uso di metodi del 2005, optando invece per i nuovi paradigmi introdotti nel 2017

It is important to note that the criteria for an etiologic diagnosis of Lewy body disease is different across versions of
the UDS:
• In v1.2 and v2 the McKeith 2005 criteria were applied 
• In v3 and v4 the McKeith 2017 are applied

In [ ]:

df_tmp = df.sort_values(by=['NACCID', 'NACCVNUM'])

df_trasversal = df.drop_duplicates(subset=['NACCID'], keep='first').copy()

# Supponiamo che df_trasversal sia il tuo dataframe già filtrato per avere solo la visita 1

# 1. Verifichiamo quali versioni UDS abbiamo nel dataset e quanti pazienti ci sono per versione
print("Distribuzione versioni UDS prima del filtro:")
print(df_trasversal['FORMVER'].value_counts(dropna=False))

# 2. Applichiamo il taglio: teniamo solo le versioni 3 (che include 3.0) e 4
# Convertiamo prima la colonna in stringa per evitare problemi tra float 3.0 e int 3
df_trasversal['FORMVER'] = df_trasversal['FORMVER'].astype(str)

# Filtriamo usando una condizione booleana
df_lastestCriteria = df_trasversal[df_trasversal['FORMVER'].str.startswith(('3', '4'))].copy()
    

print(f"\nPazienti totali rimasti (UDS v3 e v4): {len(df_lastestCriteria)}")

# Ora il tuo df_moderno è il nuovo punto di partenza!
# Da qui in poi, puoi applicare lo script di pulizia (blank, -4, 999, soglia 50%) 
df = df_lastestCriteria.copy()

del df_tmp, df_trasversal, df_lastestCriteria


In [ ]:


# Salviamo l'elenco delle colonne PRIMA di qualsiasi modifica
colonne_prima_del_taglio = set(df.columns)

# PULIZIA VALORI VUOTI E CODICI ASSENTI
# Sostituiamo stringhe vuote con NaN (nota l'uso di inplace=True)
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

lbdsynt = df['LBDSYNT'].copy()  # Salviamo i sintomi LBD prima di eventuali modifiche


# RIMOZIONE INTELLIGENTE DEGLI 8 E 9 (Solo nelle colonne categoriali/ordinali)
# Le scale NACC di solito vanno da 0 a 1 (binarie), o da 0 a 3/4/5 (gravità).
# Definiamo i valori validi attesi per queste scale, aggiungendo 8 e 9.
valori_scala_clinica = {0, 1, 2, 3, 4, 5, 8, 9, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 8.0, 9.0}
# Sostituiamo i codici clinici "falsi" con veri NaN
codici_mancanti = [-4, 777, 888, 999, 8888, 99, 88, 888.8, 88.8]

#colonna_age = df["NACCAGE"].copy()
df.mask(df.isin(codici_mancanti), np.nan, inplace=True)
# preserviamo l'età nella stessa colonna o in backup se necessario
#df['NACCAGE'] = colonna_age

colonne_modificate_valori_binari = 0

for col in df.columns:
    # Estraiamo i valori unici della colonna, ignorando i NaN già presenti
    valori_unici = set(df[col].dropna().unique())
    
    # Se i valori unici della colonna sono un SOTTOINSIEME della nostra scala...
    # (es. se la colonna ha solo {0, 1, 9} rientrerà in questo caso)
    if valori_unici.issubset(valori_scala_clinica) and (9 in valori_unici or 9.0 in valori_unici):
        # ...allora sostituiamo in modo sicuro!
        df[col] = df[col].replace([8, 9, 8.0, 9.0], np.nan)
        colonne_modificate_valori_binari += 1

print(f"Sostituiti 8/9 con NaN in {colonne_modificate_valori_binari} colonne categoriali.")

# TAGLIO DELLE COLONNE CON TROPPI NaN (Soglia 70%)
threshold_percentage = 0.70
# thresh richiede un intero: numero minimo di non-NaN richiesti per conservare la colonna
threshold = int(len(df) * threshold_percentage)
df = df.dropna(axis=1, thresh=threshold)


# FILTRO NEAR-ZERO VARIANCE
soglia_polarizzazione = 0.85 
colonne_da_rimuovere_polarizzate = []

for col in df.columns:
    # value_counts(normalize=True) calcola le percentuali ignorando i NaN
    if not df[col].dropna().empty:
        # Prende la frequenza del valore più comune
        frequenza_max = df[col].value_counts(normalize=True).max()
        
        if frequenza_max > soglia_polarizzazione:
            colonne_da_rimuovere_polarizzate.append(col)



# Prima di eliminare, salviamo età e scolarità se per caso finissero qui dentro (improbabile, ma sicuro)
colonne_da_rimuovere_polarizzate = [c for c in colonne_da_rimuovere_polarizzate if c not in ['NACCAGE', 'EDUC', 'LBDSYNT']]



# RIMOZIONE COLONNE AMMINISTRATIVE
parole_chiave_admin = ['DATE', 'VISIT', 'PACKET', 'FORM', 'VERSION', 'SOURCE', 'VST', 'RACE', 'ETH', 'INSEX', 'INRASEC', 'INEDU', 'INHISP',
                       'INHISPOR', 'INHISPOX', 'INRATER', 'ADG', 'NGD', 'EDU', 'DAYS', 'FDYS', 'BIRTH', 'INREL', 'DRUG', 'HTN'
                       ]

key_to_save = ["KIDNEY", "NACCLBD", "PCA", "LBDSYNT", "PSPSYN", "NACCUDSD"]
colonne_via_substring = [col for col in df.columns if any(keyword in col for keyword in parole_chiave_admin)]
colonne_da_preservare = [col for col in df.columns if any(keyword in col for keyword in key_to_save)]

#NACCADMD farmaci contro alz
nomi_esatti_admin = [
    'NACCVNUM', 'NACCNURP', 'NACCPAFF', 'NACCNINR', 'NACCADC', 'NACCMOD', 'NACCACTV', 'NACCNOVS', 'NACCDSDY', 'NACCDSMO', 'NACCMDSS',
    'NACCAGEB', 'NACCNHIR', 'MARISTAT', 'NACCLIVS', 'NACCREFR', 'INLIVWTH', 'INCALLS', 'INBIRMO', 'INBIRYR', 'NACCAMD', 'NACCACEI', 'NACCDIUR',
    'NACCNSD', 'NACCADMD', 'NACCAGEB', 'NACCDIED', 'NACCAUTP', 'NACCLDBM', 'NACCAPOE', 'NACCNE4S', 'NACCNCRD', 'NACCETPR', 'COMMUN', 'DIABETES', 'HYPERTEN',
    'THYDIS', 'DECCLMOT', 'IMAGMACH' 
]


# Uniamo tutte le colonne candidate alla rimozione e quindi escludiamo quelle critiche da preservare.
colonne_da_rimuovere = list(
    set(colonne_via_substring + nomi_esatti_admin + colonne_da_rimuovere_polarizzate)
    - set(colonne_da_preservare)
 )


if 'NACCID' in df.columns:
    colonne_da_rimuovere.append('NACCID')

# Applichiamo l'eliminazione
df = df.drop(columns=colonne_da_rimuovere, errors='ignore')


df['LBDSYNT'] = lbdsynt  # Ripristiniamo i sintomi LBD nella colonna originale
# 6. REPORT FINALE
colonne_dopo_il_taglio = set(df.columns)
colonne_eliminate = colonne_prima_del_taglio - colonne_dopo_il_taglio

print(f"Dimensione matrice X finale: {df.shape}")
print(f"Sono state eliminate {len(colonne_eliminate)} colonne in totale.")
print("-" * 40)

# Stampiamo le colonne eliminate per verifica
for col in sorted(colonne_eliminate):
    print(f"- {col}")

In [ ]:


# 1. Classe 0: Sani (Controllo)
# Dal manuale: "Participants without cognitive or behavioral impairment have NACCLBDS = 8"
cond_sani = (df['NACCLBDS'] == 8)
# 2. Classe 2: Demenza a Corpi di Lewy (DLB)
# Dal manuale: NACCLBDS = 1 significa LBD presente.
cond_dlb = (df['NACCLBDE'] == 1) & (df['NACCUDSD'] == 4) 
# Aggiungiamo NACCUDSD == 4 per essere certi che sia allo stadio di "Demenza" (escludendo MCI)
cond_dlb = (df['NACCLBDE'] == 1 or df['LBDSYNT'] == 1) & (df['NACCUDSD'] == 4)

# 3. Classe 1: Alzheimer Puro (AD)
# NACCUDSD == 4 (Demenza) AND NACCALZD == 1 (Eziologia Alzheimer)
# Aggiungiamo NACCLBDS == 0 per assicurarci che NON abbia anche i Corpi di Lewy (No Demenza Mista)
cond_ad = (df['NACCUDSD'] == 4) & (df['NACCALZD'] == 1) & (df['NACCLBDS'] == 0)


# Applichiamo la mappatura
condizioni = [cond_sani, cond_ad, cond_dlb]
etichette = [0, 1, 2] # 0=Sani, 1=Alzheimer, 2=Lewy Body
df['TARGET'] = np.select(condizioni, etichette, default=np.nan)

# Eliminiamo tutti i casi non chiari, le demenze miste e altri tipi di demenza (frontotemporale, vascolare...)
df.dropna(subset=['TARGET'], inplace=True)


# Verifichiamo il bilanciamento delle nostre tre classi!
print("Distribuzione delle diagnosi nel dataset finale:")
print(df['TARGET'].value_counts())



# Isoliamo la y (Target)
y = df['TARGET'].copy()

# Estraiamo la X (Feature) scartando la colonna TARGET e TUTTE le colonne diagnostiche
# Sostituisci la lista con i prefissi esatti delle colonne del Form D del tuo Parquet

target_keywords = ['ALZ', 'LBD', 'UDSD', 'DLB', 'ADMD', 'NACC']
colonne_da_eliminare = colonne_via_substring = [col for col in df.columns if any(keyword in col for keyword in target_keywords) and col not in ['NACCAGE']]


df.drop(columns=colonne_da_eliminare, inplace=True)

X = df.copy()

In [ ]:


# 1. Calcoliamo la matrice di correlazione di Kendall
# Usiamo numeric_only=True per evitare errori con eventuali colonne testuali residue
matrice_correlazione = df.corr(numeric_only=True, method='kendall')


fig = px.imshow(
    matrice_correlazione,
    text_auto=".2f",                 # Mostra i numeri arrotondati a 2 decimali
    aspect="auto",                   # Adatta la forma allo schermo
    color_continuous_scale="RdBu_r", # Scala cromatica classica: Rosso (negativa) / Blu (positiva)
    zmin=-1, zmax=1,                 # Fissa i limiti della correlazione tra -1 e 1
    title="Matrice di Correlazione Interattiva (Metodo Kendall)"
)

# 3. Miglioriamo l'estetica per i notebook
fig.update_layout(
    width=900, 
    height=800,
    xaxis_tickangle=-45 # Inclina le etichette per renderle leggibili
)

# Mostriamo il grafico
fig.show()

In [ ]:
cognitive_columns = [col for col in df.columns if col.endswith('IF') or col.endswith('2F') or col.endswith('3F')]
X.drop(columns=cognitive_columns, inplace=True)

arth_columns = [col for col in df.columns if col.startswith('ART') and col not in ['ARTH']]
X.drop(columns=arth_columns, inplace=True)

In [ ]:
X

In [ ]:

ax = sns.stripplot(
    data=df, 
    x='DECCLBE', 
    y='BEMODE', 
    hue='TARGET', 
    palette='bright', 
    alpha=0.6,
    jitter=0.2,
    dodge=True  # separa fisicamente le 3 classi sull'asse X
)

handles, previous_labels = ax.get_legend_handles_labels()

# Definisci le tue nuove etichette testuali nello STESSO ORDINE dei numeri (0.0, 1.0, 2.0)
nuove_etichette = ['Sani', 'Alzheimer', 'Lewy Body']

# Ricrea la legenda assegnando gli handles originali alle nuove stringhe
ax.legend(
    handles=handles, 
    labels=nuove_etichette, 
    title='Significato Classi', 
    bbox_to_anchor=(1.05, 0.5), 
    loc='center left'
)

sns.despine()
plt.show()

In [ ]:
p = sns.stripplot(
    data=df,
    x='DEMENTED',
    y='PCA',
    hue='TARGET',
    palette='bright',
    jitter=0.2,
    dodge=True
)

handles, previous_labels = p.get_legend_handles_labels()

p.legend(
    handles=handles, 
    labels=nuove_etichette, 
    title='Significato Classi', 
    bbox_to_anchor=(1.05, 0.5), 
    loc='center left'
)
sns.despine()
plt.show()


In [ ]:
p = sns.stripplot(
    data=df,
    x='DEMENTED',
    y='NAMNDEM',
    hue='TARGET',
    palette='bright',
    jitter=0.2,
    dodge=True
)

handles, previous_labels = p.get_legend_handles_labels()

p.legend(
    handles=handles, 
    labels=nuove_etichette, 
    title='Significato Classi', 
    bbox_to_anchor=(1.05, 0.5), 
    loc='center left'
)
sns.despine()
plt.show()

In [ ]:
p = sns.barplot(
    data=df,
    x='MRFTLD',
    y='HIPPATR',
    hue='TARGET',
    palette='bright',
    dodge=True
)

handles, previous_labels = p.get_legend_handles_labels()

p.legend(
    handles=handles, 
    labels=nuove_etichette, 
    title='Significato Classi', 
    bbox_to_anchor=(1.05, 0.5), 
    loc='center left'
)
sns.despine()
plt.show()

In [ ]:
colonne_da_escludere = ['DECCLBE', 'BEMODE', 'NAMNDEM', 'PCA', 'AMNDEM', 'MRFTLD'] #MRFTLD fa riferimento a screeing image
X.drop(columns=colonne_da_escludere, inplace=True)


In [ ]:
df = X.copy()
df['TARGET'] = y.copy()  # Aggiungiamo la colonna TARGET al dataframe df per le analisi successive

In [ ]:
from utils import ASSETS_DIR
print(f"Salvataggio del dataframe pulito in: {ASSETS_DIR / 'investigator_ftldlbd_nacc74_cleaned.parquet'}")
if not ASSETS_DIR.exists():
    ASSETS_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(ASSETS_DIR / 'nacc74_cleaned.parquet', index=False)